# Interaction discovery

This notebook is part of the staged manuscript-reproduction workflow. During automated validation it resolves placeholder paths to the deterministic demo dataset; once local real-data paths are configured it will run against the real case-study artifacts.

In [ ]:
import os
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    env_root = os.environ.get("BSM_PUBLIC_RF_REPO_ROOT")
    if env_root:
        candidate = Path(env_root).resolve()
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    raise RuntimeError("Could not locate the bsm-public-rf repository root.")


REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from bsm_rfm import (
    build_manuscript_notebook_context,
    run_interaction_discovery_stage,
)

context = build_manuscript_notebook_context(REPO_ROOT, "04_interaction_discovery.ipynb")
stage_result = run_interaction_discovery_stage(context)
stage_result.interactions.summary

### Step 1: Generate Interaction Features

Create interaction and polynomial terms.

**Process:**
1. Select features to interact (all or top-N by significance)
2. Generate polynomial expansions: x², x³, etc. (up to configured degree)
3. Generate pairwise interactions: x_i * x_j for selected pairs
4. Remove constant and highly collinear interactions

**Parameters:**
- `polynomial_degree`: Max polynomial degree (default 2)
- `interaction_sparsity`: Only interact top-N features (default 50)
- `interaction_method`: 'all-pairs', 'targeted', or 'pairwise-correlation'

**Output:**
- `interaction_features_df`: New interaction feature matrix
- Typically adds 100-500 new features

### Step 2: Combine Original and Interaction Features

Merge original features with new interaction/polynomial features.

**Output:**
- Full feature matrix: original + interactions (~200-700 features)
- Feature metadata: which features are interactions, which are originals